# RAG Evaluation with RAGAS + Gemini

This notebook builds a small RAG pipeline (Qdrant + Gemini) and evaluates it with
[RAGAS](https://docs.ragas.io) using Google Gemini as both the generator LLM and
the RAGAS evaluator LLM.

**Fixed in this version:**
- Removed duplicate / conflicting `pip install` cells and pinned versions that
  are known to install together without dependency conflicts.
- Fixed the incomplete `build_prompt` function (was missing `:` and a body,
  which caused a `SyntaxError` when the cell ran).
- Removed the unused OpenAI imports/functions (`langchain_openai`, `openai`)
  since the whole pipeline now runs on Gemini only.
- Consolidated the many repeated "RAGAS LLM/embeddings" cells into a single
  definition.
- Fixed `list(client.list_examples(...))[0]` being called three separate times
  (three separate API calls) → now called once and reused.
- Added `nest_asyncio` so the `await` calls work reliably inside Jupyter no
  matter which kernel/event loop is running.


## 1. Install dependencies

These versions were verified together in a clean virtual environment
(`pip check` reports **no broken requirements**):

| package | version |
|---|---|
| ragas | 0.3.7 |
| langchain | 0.3.30 |
| langchain-core | 0.3.86 |
| langchain-community | 0.3.31 |
| langchain-google-genai | 2.1.12 |
| google-genai | latest 2.x |
| qdrant-client | latest 1.x |
| langsmith | latest |
| python-dotenv | latest |

> ⚠️ Do **not** `pip install -U langchain langchain-google-genai` on top of this.
> The newest `langchain-google-genai` (4.x) requires `langchain-core>=1.4`,
> which is incompatible with `ragas==0.3.7` (which still depends on the
> `langchain-community` 0.3.x line). Mixing the two is what caused the
> `ModuleNotFoundError: No module named 'langchain_community.chat_models.vertexai'`
> style errors in the original notebook.


In [2]:
! uv pip install -q "ragas==0.3.7" \
    "langchain==0.3.30" \
    "langchain-core==0.3.86" \
    "langchain-community==0.3.31" \
    "langchain-google-genai==2.1.12" \
    "google-genai" \
    "qdrant-client" \
    "langsmith" \
    "python-dotenv" \
    "nest_asyncio"


## 2. Imports

In [3]:
import os
import asyncio

import nest_asyncio
from dotenv import load_dotenv

# Google & Qdrant
from google import genai
from google.genai import types
from qdrant_client import QdrantClient

# LangSmith
from langsmith import Client

# LangChain (Gemini) wrappers used by RAGAS
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# RAGAS
from ragas import SingleTurnSample
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    IDBasedContextPrecision,
    IDBasedContextRecall,
)

# Lets `await` work smoothly inside Jupyter regardless of the kernel's event loop
nest_asyncio.apply()


2026-07-08 21:04:30.653078956 [W:onnxruntime:Default, device_discovery.cc:164 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:89 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## 3. Environment & clients

Put a `.env` file next to this notebook containing:

```
GOOGLE_API_KEY=your-gemini-api-key
LANGSMITH_API_KEY=your-langsmith-api-key
```


In [4]:
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
if not GOOGLE_API_KEY:
    raise ValueError("GOOGLE_API_KEY not found in .env file!")

# Native google-genai client -> used for the RAG pipeline itself
# (embeddings + answer generation)
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

# LangSmith client -> used to pull the evaluation dataset
ls_client = Client()


## 4. RAGAS evaluator LLM & embeddings

RAGAS needs its own LLM (to judge faithfulness / relevancy) and its own
embedding model (for `ResponseRelevancy`). We wrap Gemini via LangChain so
RAGAS can use it.


In [12]:
# --- Available Gemini LLM models for the RAGAS judge ---
# "gemini-2.5-flash" : fast & cheap, good default for evaluation
# "gemini-2.5-pro"   : stronger reasoning, more expensive
ragas_llm = LangchainLLMWrapper(
    ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=GOOGLE_API_KEY)
)

# --- Available Gemini embedding models ---
# "models/text-embedding-004" : recommended, current-generation
ragas_embeddings = LangchainEmbeddingsWrapper(
    GoogleGenerativeAIEmbeddings(model="models/text-embedding-004", google_api_key=GOOGLE_API_KEY)
)


/tmp/ipykernel_3174707/2816489317.py:4: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  ragas_llm = LangchainLLMWrapper(
/tmp/ipykernel_3174707/2816489317.py:10: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(


## 5. Download a reference example from LangSmith

In [6]:
dataset = ls_client.read_dataset(dataset_name="rag-evaluation-dataset")

# Pull the examples ONCE and reuse them (the original notebook called
# list_examples three separate times for no reason)
examples = list(ls_client.list_examples(dataset_id=dataset.id, limit=10))

reference_input = examples[0].inputs
reference_output = examples[0].outputs

reference_input, reference_output


({'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?'},
 {'ground_truth': "I'm sorry, but the specific duration of the warranty for the Garmin 890 RV GPS Navigator is not mentioned in the product description, only that it comes with a 'Limited Warranty'.",
  'reference_context_ids': [],
  'reference_descriptions': []})

## 6. RAG pipeline

* `get_embedding` – embeds a query with `gemini-embedding-001`
* `retrieve_data` – queries Qdrant and returns ids/context/ratings/scores
* `process_context` – turns retrieved chunks into a formatted context string
* `build_prompt` – builds the final prompt for the answer-generation model
* `generate_answer` – calls Gemini to answer the question
* `rag_pipeline` – glues everything together


In [7]:
def get_embedding(text, model="gemini-embedding-001"):
    """Generate an embedding vector for `text` using a Gemini embedding model."""
    response = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(),
    )
    return response.embeddings[0].values


def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-03",
        query=query_embedding,
        limit=k,
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        retrieved_context_ratings.append(result.payload["average_rating"])
        similarity_scores.append(result.score)

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "retrieved_context_ratings": retrieved_context_ratings,
        "similarity_scores": similarity_scores,
    }


def process_context(context):
    formatted_context = ""
    for id_, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id_}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(preprocessed_context, question):
    """NOTE: this function was left incomplete (missing `:` and body) in the
    original notebook, which raised a SyntaxError. Fixed here."""
    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- You need to answer the question based on the provided context only.
- Never use the word "context" and refer to it as the available products.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


def generate_answer(prompt, model_name="gemini-2.5-flash"):
    """Generates a response using the specified Gemini model."""
    response = gemini_client.models.generate_content(
        model=model_name,
        contents=prompt,
    )
    return response.text


def rag_pipeline(question, top_k=5):
    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    return {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }


### Quick smoke test

In [8]:
rag_pipeline("Can I get some charger?", top_k=5)


{'answer': 'Yes, there are several charging cables available to choose from:\n\n*   A replacement charger cable for the Fitbit Inspire 3, which comes in a 2-pack and is 3.3ft long.\n*   A 5-in-1 multi-charging cable (3M/10Ft) that includes USB A/USB C to Lightning, Type C, and Micro USB connectors. This cable can charge multiple devices simultaneously and is compatible with various iPhones, Android phones, Kindles, and Bluetooth headsets.\n*   Several Apple MFi Certified Lightning cables for iPhones, iPads, and iPods. These include:\n    *   A 3-pack of 3ft USB A to Lightning cables.\n    *   A 6-pack of colorful nylon Lightning cables in various lengths (3ft, 6ft, 10ft).\n    *   A 2-pack of 6ft USB C to Lightning cables, designed for fast charging compatible Apple devices.',
 'question': 'Can I get some charger?',
 'retrieved_context_ids': ['B0BGDQLZD2',
  'B0BFPZGYLD',
  'B0BYYLJRHT',
  'B09TNXY54Y',
  'B0BV6PWVCG'],
 'retrieved_context': ['Mixblu Charger Cable Replacement for Fitbi

## 7. Run the pipeline on the LangSmith reference question

In [9]:
result = rag_pipeline(reference_input["question"])
result


{'answer': 'Based on the available products, the Garmin 890 8-inch RV GPS Navigator comes with a limited warranty, but the specific duration of this warranty is not mentioned.',
 'question': 'What is the warranty period for the Garmin 890 RV GPS Navigator?',
 'retrieved_context_ids': ['B08BX2L8F2',
  'B0B3MMP22L',
  'B0BLV7YH2W',
  'B0BB1YKXL1',
  'B09QGNB537'],
 'retrieved_context': ['Garmin 890 8-inch RV GPS Navigator Bundle with Car Charger Expander and Hard Shell EVA Case for Tablets/GPS (010-02425-00) Built in Wi-Fi connectivity makes updating your maps of North America a breeze. This large 8" GPS navigator features a bright, high-resolution edge-to-edge touchscreen display so you can easily see important information Built-in Wi-Fi connectivity makes it easy to keep your maps and software up to date without using a computer IN THE BOX: RV 890 | Vehicle suction cup with powered magnetic mount | Screw down mount | 1" ball adapter with AMPS plate | Vehicle power cable | USB cable | D

## 8. RAGAS metrics

Four metrics are used:

- **Faithfulness** – is the answer grounded in the retrieved context? (needs an LLM)
- **ResponseRelevancy** – is the answer relevant to the question? (needs an LLM + embeddings)
- **IDBasedContextPrecision** – of the retrieved ids, how many are in the reference ids? (no LLM needed, pure ID comparison)
- **IDBasedContextRecall** – of the reference ids, how many were retrieved? (no LLM needed, pure ID comparison)


In [13]:
async def ragas_faithfulness(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = Faithfulness(llm=ragas_llm)
    return await scorer.single_turn_ascore(sample)


async def ragas_response_relevancy(run, example):
    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)
    return await scorer.single_turn_ascore(sample)


async def ragas_context_precision_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)


async def ragas_context_recall_id_based(run, example):
    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"],
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)


## 9. Run all evaluations

In [14]:
async def run_evaluations(run, example):
    faithfulness_score = await ragas_faithfulness(run, example)
    relevancy_score = await ragas_response_relevancy(run, example)
    precision_score = await ragas_context_precision_id_based(run, example)
    recall_score = await ragas_context_recall_id_based(run, example)

    return {
        "faithfulness": faithfulness_score,
        "response_relevancy": relevancy_score,
        "id_based_context_precision": precision_score,
        "id_based_context_recall": recall_score,
    }


scores = await run_evaluations(result, reference_output)
scores


GoogleGenerativeAIError: Error embedding content: 404 models/text-embedding-004 is not found for API version v1beta, or is not supported for embedContent. Call ModelService.ListModels to see the list of available models and their supported methods.